In [1]:
import pandas as pd
import numpy as np

复杂查询

In [ ]:
df=pd.DataFrame(data=np.random.randint(50,100,(50,4)),columns=['Q1','Q2','Q3','Q4'])
df['team']=np.random.choice(['A','B','C'],size=(50),p=[0.4,0.3,0.3])
df['id']=np.arange(1,51)
df.head()

,Q1,Q2,Q3,Q4,team,id
0,92,72,96,77,C,1
1,59,84,59,87,B,2
2,99,90,78,55,B,3
3,97,86,73,89,C,4
4,50,73,82,59,A,5


In [3]:
df.Q1>60

0     False
1      True
2     False
3      True
4      True
5      True
6      True
7      True
8      True
9     False
10    False
11     True
12     True
13     True
14     True
15    False
16     True
17     True
18     True
19     True
20     True
21     True
22     True
23     True
24     True
25    False
26     True
27     True
28     True
29    False
30     True
31     True
32    False
33     True
34     True
35     True
36     True
37     True
38    False
39     True
40     True
41     True
42    False
43    False
44    False
45     True
46     True
47    False
48     True
49     True
Name: Q1, dtype: bool

In [ ]:
df.loc[:,'Q1':'Q2']>70

In [ ]:
(~df.Q1>70) & (df.team=='C')

0     False
1     False
2     False
3     False
4     False
5     False
6     False
7     False
8     False
9     False
10    False
11    False
12    False
13    False
14    False
15    False
16    False
17    False
18    False
19    False
20    False
21    False
22    False
23    False
24    False
25    False
26    False
27    False
28    False
29    False
30    False
31    False
32    False
33    False
34    False
35    False
36    False
37    False
38    False
39    False
40    False
41    False
42    False
43    False
44    False
45    False
46    False
47    False
48    False
49    False
dtype: bool

数据类型转换

推断类型
- infer_objects 自动转换合适的类型
- convert_dtype 新方法 支持string转换

类型转换
1. pd.to_xxx讲数据安全转换,error参数实现兜底,downcast向下兼容实现内存缩小
error
- raise 无任何兜底,直接报错
- coerce(强制) 遇到无法转换变成Nan,后续通过dropna与fillna做二次处理
- ignore 忽略不兜底,遇到错误原封不动

downcast
- integer int8/16/32/64
- signed 同时
- unsigned 降低为无符号整数最小类型 unit8/16/32/64
- float 降低浮点数最小类型 float32/float64
> 需要注意coerce的配合,如果数据Nan失效


```python
# 通过这样转换
df=df.select_dtype(include='number')
df.apply(pd.to_numeric)
```

2. astype
直接在对应列上转换,或者传入列字典
```python
df.Q1.astype('int32').dtypes
df.astype({'Q1':'int32'})
```

In [4]:
df.dtypes

Q1       int64
Q2       int64
Q3       int64
Q4       int64
team    object
dtype: object

In [6]:
df.infer_objects().dtypes

Q1       int64
Q2       int64
Q3       int64
Q4       int64
team    object
dtype: object

In [7]:
df.convert_dtypes().dtypes

Q1               Int64
Q2               Int64
Q3               Int64
Q4               Int64
team    string[python]
dtype: object

In [20]:
pd.set_option('mode.copy_on_write',True)
Q=df['Q1']
print('优化前内存占用',Q.memory_usage(),Q.dtype)
pd.to_numeric(Q,downcast='integer')
print('优化后内存占用',Q.memory_usage(),Q.dtype)


Q=Q.astype('int16')
print('as 优化后内存占用',Q.memory_usage(),Q.dtype)

优化前内存占用 532 int64
优化后内存占用 532 int64
as 优化后内存占用 232 int16


In [ ]:
df.Q1.apply(lambda x:x+100)
df

索引排序
sort_index 生序排序,ascending=False降序,axis=1按列,na_position='first'空值在前面

reindex按照要求指导索引排队,索引不能重复.另外index用于行,columns用于列,不能与axis与上面联合使用.axis配合labels


数值排序
sort_values按照数字大小,字符按字母;dataframe可以传入多个列,每列也可以用ascending

混合排序
索引和数值排序,sort_values(by=['idx_name','col']),map排序如果第一列相同,在根据第二列大小进行排序

按值大小排序
nsmallest(num,[col])
nlargest(num,[col])

In [ ]:
# ignore_index
df_id=df.set_index('id')
df_id.sort_index(ascending=False,ignore_index=True)


df1=pd.DataFrame({'A':[1,2,3,5],'B':[2,4,5,5]},index=['a','b','c','d'])
# df1.reindex(labels=['b','c','a','d'],axis=0)
df1.reindex(labels=['B','A'],axis=1)

,B,A
a,2,1
b,4,2
c,5,3
d,5,5


In [58]:
df.index

RangeIndex(start=0, stop=50, step=1)

In [61]:
idx_df=df.set_index('id')
idx_df.sort_values(by=['id','Q1'],ascending=[False,False]).head()

,Q1,Q2,Q3,Q4,team
id,,,,,
50,79,61,70,92,C
49,79,79,77,83,B
48,90,74,74,90,C
47,85,98,98,81,C
46,58,91,92,51,B


In [63]:
idx_df.nlargest(n=5,columns=['Q1','Q2'])

,Q1,Q2,Q3,Q4,team
id,,,,,
3,99,90,78,55,B
13,99,78,88,75,C
39,99,59,75,71,C
21,98,61,60,73,A
4,97,86,73,89,C


添加修改

1. 修改数值
df.loc[0,0]=1 直接筛选出要修改的数据范围再赋值就可以

2. replace 方法
可以批量替换,replace[[1,2],3]把1,2替换成3,replace[[1,2],[4,5]]把1,2替换成4,5,replace({'col1':rowno,},val)指定替换,replace('Q1',{'old':'new'});可以使用参数regex正则表达式替换

3. fillna
对空值填入特定数据,value可以传字典对应不同列填不同值
bfill将空值修改为下一个值,ffill将空值修改为上一个值,

4. 修改索引名
df.index/columns直接赋值修改,rename(colomns={'col':'val'})

5. 增加列
新增加列与修改数值一样都是实时生效
    - 直接定义df['new']=val;通过insert添加参数loc第几列,column,value列值,allow_dupicates允许相同列

    - assign 增加临时列不改变原有数据,方便查看不会造成数据混乱;可链式编程

    -  eval 可以传入字符表达式进行新增列

6. 增加行,使用loc[max]

7. 合并追加
pd.concat([]),ignore_index忽略索引,keys变为多层索引

8. 删除
pop可删除单个,也可删除列 todo drop

9. 删除空值 dropno
    - axis=columns 只保留全有值列
    - how=all 行或列全没值才删除
    - thresh 只少有两个空值才删
    - inplace 删除并替换生效

In [110]:
df2=pd.DataFrame({'A':5,'B':pd.date_range('2024-01',freq='ME',periods=10)
                  ,'C':np.random.randint(10,20,(10,))})
df2.A.astype('object')
df2.iloc[0:4,0]=None
df2.head()

,A,B,C
0,NaN,2024-01-31,13
1,NaN,2024-02-29,17
2,NaN,2024-03-31,13
3,NaN,2024-04-30,18
4,5.0,2024-05-31,17


In [ ]:
df2.replace({'A':5.0},20,inplace=True)
df2.replace([14,10],[1,1])

df2.iloc[0,0]=1
df2.ffill()

In [ ]:
df2.rename(columns={'A':'AA'})

In [ ]:
df2['10+C']=10+df2['C']
df2.insert(0,'A',1,allow_duplicates=True)
df2

In [102]:
df2.assign(D=2).assign(R=4)

,A,A,A,B,C,A+C,10+C,D,R
0,1,1,1.0,2024-01-31,14,15.0,24,2,4
1,1,1,NaN,2024-02-29,13,NaN,23,2,4
2,1,1,1.0,2024-03-31,19,20.0,29,2,4
3,1,1,NaN,2024-04-30,14,NaN,24,2,4
4,1,1,20.0,2024-05-31,19,39.0,29,2,4
5,1,1,20.0,2024-06-30,14,34.0,24,2,4
6,1,1,20.0,2024-07-31,10,30.0,20,2,4
7,1,1,20.0,2024-08-31,17,37.0,27,2,4
8,1,1,20.0,2024-09-30,15,35.0,25,2,4
9,1,1,20.0,2024-10-31,11,31.0,21,2,4


In [111]:
df3=pd.DataFrame({'A':5,'B':pd.date_range('2024-01',freq='ME',periods=10)
                  ,'C':np.random.randint(10,20,(10,))})

df3.eval('AA=A+C')

,A,B,C,AA
0,5,2024-01-31,10,15
1,5,2024-02-29,18,23
2,5,2024-03-31,13,18
3,5,2024-04-30,17,22
4,5,2024-05-31,14,19
5,5,2024-06-30,17,22
6,5,2024-07-31,17,22
7,5,2024-08-31,15,20
8,5,2024-09-30,12,17
9,5,2024-10-31,14,19


In [116]:
s1=pd.Series(['a','b'])
s2=pd.Series(['c','d'])
pd.concat([s1,s2],ignore_index=True)

0    a
1    b
2    c
3    d
dtype: object

In [117]:
pd.concat([s1,s2],keys=['s1','s2'])

s1  0    a
    1    b
s2  0    c
    1    d
dtype: object

高级过滤
- select_dtypes(include='number') 只保留数字类型
- where(bool/lambda/)不满足返回None,可以将不满足的替换为另外一个值(可以用np.where)
> 不能像np.where(exp,val1,val)满足val1,不满足用val
- mask与where一样不同的是满足用Nan填充

In [124]:
df3=pd.DataFrame({'A':5,'B':pd.date_range('2024-01',freq='ME',periods=10)
                  ,'C':np.random.randint(10,100,(10,))})

In [122]:
df3.select_dtypes(include='number')

,A,C
0,5,10
1,5,18
2,5,13
3,5,17
4,5,14
5,5,17
6,5,17
7,5,15
8,5,12
9,5,14


In [129]:
df3.where(df3.C>60,0)

,A,B,C
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
5,5,2024-06-30 00:00:00,91
6,5,2024-07-31 00:00:00,96
7,0,0,0
8,0,0,0
9,5,2024-10-31 00:00:00,75


In [133]:
df3.assign(passed=lambda x:np.where(x.C>60,'yes','no'))

,A,B,C,passed
0,5,2024-01-31,48,no
1,5,2024-02-29,36,no
2,5,2024-03-31,24,no
3,5,2024-04-30,21,no
4,5,2024-05-31,48,no
5,5,2024-06-30,91,yes
6,5,2024-07-31,96,yes
7,5,2024-08-31,26,no
8,5,2024-09-30,29,no
9,5,2024-10-31,75,yes


In [ ]:
df3.C.mask(df3.C>70,100)

0     48
1     36
2     24
3     21
4     48
5    100
6    100
7     26
8     29
9    100
Name: C, dtype: int64

数据迭代
series迭代对象,df迭代返回的是列

- iterrows 拆包idx索引,row行
- itertuples() 返回行元组
- items



In [136]:
for _ in df3:
    print(_)

A
B
C


In [140]:
for idx,row in df3.iterrows():
    print(idx,row)

0 A                      5
B    2024-01-31 00:00:00
C                     48
Name: 0, dtype: object
1 A                      5
B    2024-02-29 00:00:00
C                     36
Name: 1, dtype: object
2 A                      5
B    2024-03-31 00:00:00
C                     24
Name: 2, dtype: object
3 A                      5
B    2024-04-30 00:00:00
C                     21
Name: 3, dtype: object
4 A                      5
B    2024-05-31 00:00:00
C                     48
Name: 4, dtype: object
5 A                      5
B    2024-06-30 00:00:00
C                     91
Name: 5, dtype: object
6 A                      5
B    2024-07-31 00:00:00
C                     96
Name: 6, dtype: object
7 A                      5
B    2024-08-31 00:00:00
C                     26
Name: 7, dtype: object
8 A                      5
B    2024-09-30 00:00:00
C                     29
Name: 8, dtype: object
9 A                      5
B    2024-10-31 00:00:00
C                     75
Name: 9, dtype: object


In [142]:
for _ in df3.itertuples():
    print(_)

Pandas(Index=0, A=5, B=Timestamp('2024-01-31 00:00:00'), C=48)
Pandas(Index=1, A=5, B=Timestamp('2024-02-29 00:00:00'), C=36)
Pandas(Index=2, A=5, B=Timestamp('2024-03-31 00:00:00'), C=24)
Pandas(Index=3, A=5, B=Timestamp('2024-04-30 00:00:00'), C=21)
Pandas(Index=4, A=5, B=Timestamp('2024-05-31 00:00:00'), C=48)
Pandas(Index=5, A=5, B=Timestamp('2024-06-30 00:00:00'), C=91)
Pandas(Index=6, A=5, B=Timestamp('2024-07-31 00:00:00'), C=96)
Pandas(Index=7, A=5, B=Timestamp('2024-08-31 00:00:00'), C=26)
Pandas(Index=8, A=5, B=Timestamp('2024-09-30 00:00:00'), C=29)
Pandas(Index=9, A=5, B=Timestamp('2024-10-31 00:00:00'), C=75)


函数应用

- pipe(<func_name>,<参数列表或字典>)函数是将df和s作为第一个参数
- apply可以按行按列,进行函数处理
- applymap对df中所有元素进行函数处理
- map 只能用于一列,可以是字典进行替换,可以是函数
- agg 汇总操作,函数或者字符[min,max,mean]
- transform 调用函数并返回一个与自身长度相同的数据
- copy 返回一个新对象与原来没关系,deep=False尽复制对数据的引用

In [145]:
df3.C.apply(lambda x:x+1)
df3.C.apply(np.mean)

0    48.0
1    36.0
2    24.0
3    21.0
4    48.0
5    91.0
6    96.0
7    26.0
8    29.0
9    75.0
Name: C, dtype: float64

In [146]:
df4=pd.DataFrame({'A':np.random.randint(40,100,(10,)),'B':np.random.choice(['a','b','c'],size=(10,))})

In [149]:
df4.B.map({'a':'aa','b':'bb','c':'cc'})
df4.B.map('is {}'.format)

0    is a
1    is b
2    is c
3    is a
4    is a
5    is b
6    is b
7    is a
8    is a
9    is c
Name: B, dtype: object

In [152]:
df4.assign(C=df4.A).agg({'A':'max','C':'min'})

A    99
C    48
dtype: int64

In [154]:
df4.assign(C=df4.A).agg(m=('A',np.max),d=('C',min))

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1986/2733675947.py:1: FutureWarning: The provided callable <function max at 0x1064db600> is currently using Series.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  df4.assign(C=df4.A).agg(m=('A',np.max),d=('C',min))
/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1986/2733675947.py:1: FutureWarning: The provided callable <built-in function min> is currently using Series.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  df4.assign(C=df4.A).agg(m=('A',np.max),d=('C',min))


,A,C
m,99.0,NaN
d,NaN,48.0


In [160]:
df4.groupby('B').transform(np.sum)

/var/folders/f7/rx7cmvdx6nl86z_s7s86w4zm0000gn/T/ipykernel_1986/501375547.py:1: FutureWarning: The provided callable <function sum at 0x1064daf20> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df4.groupby('B').transform(np.sum)


,A
0,397
1,207
2,125
3,397
4,397
5,207
6,207
7,397
8,397
9,125
